# 03 - Heterogeneous treatment effects

This notebook estimates conditional treatment effects where treatment impact varies by features.


## Causal setup

- Causal question: which units benefit more from treatment?
- Treatment: binary indicator `treatment`.
- Outcome: `outcome`.
- Covariates: `age`, `risk_score`, `prior_usage`.
- Unit: each synthetic individual.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import matplotlib.pyplot as plt

from causal_inference_lab.data_generators import make_heterogeneous_treatment_data
from causal_inference_lab.meta_learners import SMetaLearner, TMetaLearner, XMetaLearner
from causal_inference_lab.plotting import plot_cate_recovery


In [ ]:
dataset = make_heterogeneous_treatment_data(n=5_000, seed=10)
data = dataset.data
covariates = ["age", "risk_score", "prior_usage"]
x = data[covariates]
y = data["outcome"]
t = data["treatment"]

print(f"True ATE in generator: {dataset.true_ate:.3f}")
print(f"Sample size: {len(data)}")
print(f"Treated share: {t.mean():.3f}")


## T-learner CATE estimate

A T-learner fits separate models in treated and control groups and takes the difference in counterfactual predictions.


In [ ]:
t_learner = TMetaLearner().fit(
    data=data,
    covariates=covariates,
    treatment_col="treatment",
    outcome_col="outcome",
 )
cate_t = t_learner.predict_cate(x)
ate_t = t_learner.estimate_ate(x)
print(f"T-learner estimated ATE: {ate_t:.3f}")


## CATE recovery check

The synthetic generator exposes `true_ite`, so we can visually compare recovered CATE against truth.


In [ ]:
fig = plot_cate_recovery(data["true_ite"], cate_t)
plt.show()


## Meta-learner comparison (S/T/X)

S-, T-, and X-learners have different extrapolation behavior. No model should be treated as a universal truth.


In [ ]:
s_learner = SMetaLearner().fit(
    data=data,
    covariates=covariates,
    treatment_col="treatment",
    outcome_col="outcome",
)
x_learner = XMetaLearner().fit(
    data=data,
    covariates=covariates,
    treatment_col="treatment",
    outcome_col="outcome",
)

print(f"S-learner ATE: {s_learner.estimate_ate(x):.3f}")
print(f"T-learner ATE: {ate_t:.3f}")
print(f"X-learner ATE: {x_learner.estimate_ate(x):.3f}")


**Interpretation.** Heterogeneous effects are actionable for targeting, but only if the model is externally validated.
The biggest risk is treating noisy CATE estimates as exact prescriptions.
